# Solar-to-methane: an auditable calculation walkthrough

This notebook builds the toy digital twin in the same order as the physics:

1. convert 1 kg of methane into material requirements;
2. calculate each heat and electrical duty;
3. assemble the full-load specific requirement, E_req;
4. convert a solar profile into a nominal methane rate;
5. show how the operating strategy changes hourly demand;
6. size short- and long-duration storage;
7. compare perfect, imperfect forecast, and seeded faulted dispatch.

functions.py contains only equation-level calculations. model.py contains weather, schedules, stateful dispatch, economics, and case reporting. The explicit calculations below are cross-checked against the integrated model.

> All defaults are illustrative and unsuitable for investment decisions. Internal units are kW, kWh, kg, h, K, bar, and USD; timestamps are UTC.

## 1. Imports and editable assumptions

The notebook defaults to deterministic synthetic weather so that it restarts and runs without network access. Replace the weather cell with model.fetch_solar_profile(...) when a real Renewables.ninja profile is wanted.

In [ ]:
from dataclasses import asdict
import pandas as pd
import plotly.express as px

import functions as calc
import model

pd.options.display.float_format = "{:,.4f}".format
plant = model.PlantParameters()
thermal = model.ThermalParameters()

# Explicit starting inventory used consistently for battery and H2 storage.
# 0.0 = empty, 0.5 = half full, and 1.0 = full at the first simulated hour.
initial_soc_fraction = 0.50
short_storage = model.StorageParameters(
    method="battery", initial_soc_fraction=initial_soc_fraction
)
battery_long_storage = model.StorageParameters(
    method="battery", self_discharge_fraction_per_h=1e-5,
    initial_soc_fraction=initial_soc_fraction,
)
hydrogen_long_storage = model.StorageParameters(
    method="hydrogen",
    self_discharge_fraction_per_h=1e-6,
    initial_soc_fraction=initial_soc_fraction,
)
{"plant": asdict(plant), "initial_soc_fraction": initial_soc_fraction}

## 2. Methane-basis stoichiometry

The Sabatier reaction is

$$\mathrm{CO_2 + 4H_2 \rightarrow CH_4 + 2H_2O}.$$

One mole of captured CO₂ is paired with one mole of circulating CaO/CaCO₃. Dividing each molar requirement by the molar mass of methane gives kg material per kg CH₄.

In [ ]:
stoich = calc.methane_stoichiometry(methane_kg=1.0)
stoich_table = pd.DataFrame.from_dict(
    stoich, orient="index", columns=["kg per kg CH4"]
)
stoich_table

## 3. Electrical duties

Electrolysis and DAC fan electricity are simple mass-specific products:

$$E_{electrolysis}=m_{H_2}\,e_{electrolyser}$$

$$E_{fan}=m_{CO_2}\,e_{fan}.$$

CO₂ compression uses equal pressure-ratio stages with intercooling. CoolProp supplies the isentropic enthalpy change; the stage work is divided by isentropic efficiency. A documented ideal-gas fallback is used only if CoolProp cannot evaluate a state.

In [ ]:
electrolysis = calc.electrolysis_duty_kwh(
    stoich["H2"], plant.electrolyser_kwh_per_kg_h2
)
dac_fan = calc.fan_duty_kwh(
    stoich["CO2"], plant.fan_kwh_per_kg_co2
)
equilibrium_pressure = calc.calcination_equilibrium_pressure_bar(
    plant.calciner_temperature_k,
    plant.calcination_delta_h_j_per_mol,
    plant.calcination_delta_s_j_per_mol_k,
    plant.gas_constant_j_per_mol_k,
)
compression_per_kg_co2, compression_warning = calc.co2_compression_work_kwh_per_kg(
    equilibrium_pressure,
    plant.co2_outlet_pressure_bar,
    plant.intercool_temperature_k,
    plant.compressor_isentropic_efficiency,
    plant.compressor_stages,
)
co2_compression = stoich["CO2"] * compression_per_kg_co2

pd.DataFrame({
    "duty": ["Electrolysis", "DAC fan", "CO2 compression"],
    "kWh per kg CH4": [electrolysis, dac_fan, co2_compression],
    "basis": [
        f'{stoich["H2"]:.4f} kg H2 × {plant.electrolyser_kwh_per_kg_h2:.1f} kWh/kg H2',
        f'{stoich["CO2"]:.4f} kg CO2 × {plant.fan_kwh_per_kg_co2:.2f} kWh/kg CO2',
        f'{stoich["CO2"]:.4f} kg CO2 × {compression_per_kg_co2:.4f} kWh/kg CO2',
    ],
})

## 4. High-temperature calcination and solids circulation

The calcination reaction heat is charged on the CO₂ basis, converted from the molar enthalpy of reaction. That same enthalpy fixes the CaO/CaCO₂ equilibrium pressure below, so the energy balance and the thermodynamic limit cannot disagree. Hot CaO and CaCO₃ circulate between the carbonator and calciner. Their gross sensible heat is

$$Q_{solids}=(m_{CaO}c_{p,CaO}+m_{CaCO_3}c_{p,CaCO_3})(T_{calc}-T_{carb}).$$

Only the unrecovered fraction is an external electrical heat requirement. The model does not upgrade lower-temperature heat to serve this high-temperature duty.

Carbonation is the reverse reaction, so the same correlation also bounds the carbonator. CaO can only carbonate while the equilibrium CO₂ pressure stays below the partial pressure the feed gas actually offers, which inverts to

$$T_{carb}<\frac{\Delta H}{\Delta S-R\ln p_{CO_2}}.$$

Direct air capture offers only about 400 ppm, so this limit sits near 790 K — far below the temperature a CO₂-rich calcium loop would use. Above it CaCO₃ decomposes instead of forming and no capture is possible.

In [ ]:
# One enthalpy, used twice: the same delta_H that fixes the equilibrium pressure
# below also sets the calcination heat, so the two cannot be edited out of step.
calcination_kwh_per_kg_co2 = (
    plant.calcination_delta_h_j_per_mol
    / plant.carbon_dioxide_molar_mass_kg_per_mol
    / 3.6e6
)
calcination = calc.reaction_heat_kwh(stoich["CO2"], calcination_kwh_per_kg_co2)
solids = calc.solids_sensible_loss_kwh(
    stoich["CaO_circulating"],
    stoich["CaCO3_circulating"],
    plant.cao_cp_kwh_per_kg_k,
    plant.caco3_cp_kwh_per_kg_k,
    plant.carbonator_temperature_k,
    plant.calciner_temperature_k,
    plant.solids_heat_recovery_efficiency,
)
carbonation_equilibrium_pressure = calc.calcination_equilibrium_pressure_bar(
    plant.carbonator_temperature_k,
    plant.calcination_delta_h_j_per_mol,
    plant.calcination_delta_s_j_per_mol_k,
    plant.gas_constant_j_per_mol_k,
)
air_co2_partial_pressure = plant.air_co2_mole_fraction * plant.air_pressure_bar
maximum_carbonation_temperature = calc.maximum_carbonation_temperature_k(
    air_co2_partial_pressure,
    plant.calcination_delta_h_j_per_mol,
    plant.calcination_delta_s_j_per_mol_k,
    plant.gas_constant_j_per_mol_k,
)
pd.Series({
    "calcination reaction heat (kWh/kg CH4)": calcination,
    **{f"{key} (kWh/kg CH4)": value for key, value in solids.items()},
    "equilibrium CO2 pressure at calciner T (bar)": equilibrium_pressure,
    "equilibrium CO2 pressure at carbonator T (bar)": carbonation_equilibrium_pressure,
    "CO2 partial pressure in feed air (bar)": air_co2_partial_pressure,
    "carbonation driving force ratio (must exceed 1)": (
        air_co2_partial_pressure / carbonation_equilibrium_pressure
    ),
    "maximum carbonation temperature (K)": maximum_carbonation_temperature,
}, name="value").to_frame()

## 5. Feed and air heating

Two streams need preheating, and they are recovered from **separate** sources. Nothing
crosses between them.

**Carbonator air.** The DAC air stream is heated from ambient to the carbonator
temperature. The only recovery is the air/exhaust exchanger, which preheats incoming air
against the CO₂-depleted exhaust leaving the carbonator:

$$Q_{air,electric} = Q_{air,gross} \cdot (1 - \varepsilon_{hx})$$

This is by far the largest term in E_req, so `air_exhaust_hx_effectiveness` dominates the
whole energy balance.

**Sabatier feed.** The CO₂ and H₂ feed is heated from the intercool temperature to the
reactor temperature, and Sabatier reaction heat is routed directly to that feed and
nowhere else:

$$Q_{feed,electric} = \max(0,\; Q_{feed} - Q_{Sabatier})$$

There is no generic low-grade recovery factor. Any Sabatier heat left over after feed
heating is **discarded** — it is never sent to the carbonator, the calciner, or the air
preheat. At the default parameters the reaction heat more than covers the feed duty, so
the electric feed term is zero and the remainder is thrown away.

CoolProp supplies the air and feed-gas enthalpy changes, with explicit constant-property
fallbacks.


In [ ]:
air_mass = calc.dry_air_mass_for_co2_kg(
    stoich["CO2"], plant.air_co2_mole_fraction, plant.air_molar_mass_kg_per_mol,
    plant.carbon_dioxide_molar_mass_kg_per_mol, plant.dac_capture_efficiency
)
air_specific_heat, air_warning = calc.air_sensible_heat_kwh_per_kg(
    plant.reference_ambient_temperature_k,
    plant.carbonator_temperature_k,
    plant.air_pressure_bar,
    plant.air_fallback_cp_kwh_per_kg_k,
)
gross_air_heat = air_mass * air_specific_heat
carbonator_air_electric_heat = gross_air_heat * (1 - plant.air_exhaust_hx_effectiveness)
co2_feed_specific, co2_feed_warning = calc.gas_sensible_heat_kwh_per_kg(
    "CO2",
    plant.intercool_temperature_k,
    plant.sabatier_temperature_k,
    plant.co2_outlet_pressure_bar,
    plant.co2_sensible_fallback_cp_j_per_kg_k,
)
h2_feed_specific, h2_feed_warning = calc.gas_sensible_heat_kwh_per_kg(
    "H2",
    plant.intercool_temperature_k,
    plant.sabatier_temperature_k,
    plant.co2_outlet_pressure_bar,
    plant.h2_sensible_fallback_cp_j_per_kg_k,
)
gross_sabatier_feed_heat = (
    stoich["CO2"] * co2_feed_specific + stoich["H2"] * h2_feed_specific
)
sabatier_heat_to_feed = min(gross_sabatier_feed_heat, plant.sabatier_heat_kwh_per_kg_ch4)
sabatier_feed_electric_heat = max(0, gross_sabatier_feed_heat - sabatier_heat_to_feed)
sabatier_heat_discarded = max(0, plant.sabatier_heat_kwh_per_kg_ch4 - sabatier_heat_to_feed)

pd.Series({
    "dry air handled (kg/kg CH4)": air_mass,
    "gross air heat (kWh/kg CH4)": gross_air_heat,
    "carbonator air electric heat (kWh/kg CH4)": carbonator_air_electric_heat,
    "gross Sabatier feed heat (kWh/kg CH4)": gross_sabatier_feed_heat,
    "Sabatier heat routed to feed (kWh/kg CH4)": sabatier_heat_to_feed,
    "Sabatier feed electric heat (kWh/kg CH4)": sabatier_feed_electric_heat,
    "Sabatier heat discarded (kWh/kg CH4)": sabatier_heat_discarded,
}, name="value").to_frame()

### Product-storage compression and expansion

Methane leaves the process header at `co2_outlet_pressure_bar`, is compressed into the
product vessel, and is expanded to the delivery pressure. The expander recovers work but
needs interstage reheat to stay out of the two-phase region, charged as electrical
heating at COP 1 like every other heat duty here. All three terms sit inside E_req.


In [ ]:
methane_compression, _ = calc.gas_compression_work_kwh_per_kg(
    "Methane", plant.co2_outlet_pressure_bar, plant.methane_storage_pressure_bar,
    plant.intercool_temperature_k, plant.compressor_isentropic_efficiency,
    plant.storage_machine_stages, plant.methane_compression_fallback_cp_j_per_kg_k,
    plant.methane_compression_fallback_gamma,
)
methane_expansion, methane_expansion_reheat, _ = calc.gas_expansion_work_kwh_per_kg(
    "Methane", plant.methane_storage_pressure_bar, plant.methane_delivery_pressure_bar,
    plant.intercool_temperature_k, plant.storage_expander_isentropic_efficiency,
    plant.storage_machine_stages, plant.methane_compression_fallback_cp_j_per_kg_k,
    plant.methane_compression_fallback_gamma,
)
pd.Series({
    "compression": methane_compression,
    "expansion recovered": -methane_expansion,
    "expansion reheat": methane_expansion_reheat,
}, name="kWh per kg CH4").to_frame()


## 6. Assemble E_req

At full methane output, the prototype defines

$$E_{req} = E_{electrolysis} + E_{fan} + E_{compression} + Q_{calcination}
+ Q_{solids,unrecovered} + Q_{feed,electric} + Q_{air,electric}
+ W_{CH_4,storage}$$

where the last term is the product-storage package derived in section 6:
compression into the vessel, less the work the expander recovers, plus the reheat that
expansion needs. It is small - under half a percent of the total - but it belongs inside
E_req, and a hand-built breakdown that leaves it out will not reconcile with
`calculate_plant_energy`.

This is a specific electricity requirement in kWh per kg CH₄. Standby and restart heat
are strategy-dependent hourly loads and are therefore added later, not buried in E_req.

**Modelling a different capture process.** Four terms above are specific to dry calcium
looping: `dac_fan`, `calcination_reaction_heat`, `unrecovered_solids_sensible_heat` and
`carbonator_air_electric_heat`. Together they come to 24.46 kWh per kg CH₄ at the shipped
defaults, 48% of E_req, or **8.92 kWh per kg CO₂ captured** since the Sabatier
stoichiometry needs 2.743 kg CO₂ per kg CH₄. Given a published specific energy for some
other capture route, substitute it for those four: multiply your kWh/kg CO₂ figure by
2.743 and enter the result as a single breakdown term. Everything else, from electrolysis
through to the product-storage package, is capture-agnostic and stays as it is.

Doing so will break the cross-check below, because `calculate_plant_energy` still costs
the calcium loop. That is expected: either drop the assert, or move the same change onto
`PlantParameters` so both sides agree.


In [ ]:
breakdown = {
    "electrolysis": electrolysis,
    "dac_fan": dac_fan,
    "co2_compression": co2_compression,
    "calcination_reaction_heat": calcination,
    "unrecovered_solids_sensible_heat": solids["unrecovered_solids_sensible_heat"],
    "sabatier_feed_electric_heat": sabatier_feed_electric_heat,
    "carbonator_air_electric_heat": carbonator_air_electric_heat,
    "methane_storage_compression": methane_compression,
    "methane_storage_expansion_recovered": -methane_expansion,
    "methane_storage_expansion_reheat": methane_expansion_reheat,
    "sabatier_heat_recovered_to_feed": sabatier_heat_to_feed,
    "sabatier_heat_discarded": sabatier_heat_discarded,
}
e_req = calc.sum_specific_electricity_kwh_per_kg(
    breakdown, excluded_keys=("sabatier_heat_recovered_to_feed", "sabatier_heat_discarded")
)
integrated_energy = model.calculate_plant_energy(plant)

assert abs(e_req - integrated_energy.e_req_kwh_per_kg_ch4) < 1e-9
assert all(
    abs(breakdown[key] - integrated_energy.breakdown_kwh_per_kg_ch4[key]) < 1e-9
    for key in breakdown
)

energy_table = (
    pd.DataFrame.from_dict(breakdown, orient="index", columns=["kWh per kg CH4"])
    .rename_axis("component")
)
print(f"E_req = {e_req:.3f} kWh/kg CH4")
energy_table

In [ ]:
plot_energy = energy_table.drop(
    index=["sabatier_heat_recovered_to_feed", "sabatier_heat_discarded"]
).reset_index()
fig = px.bar(
    plot_energy,
    x="kWh per kg CH4",
    y="component",
    orientation="h",
    title=f"Full-load specific electricity requirement: E_req = {e_req:.2f} kWh/kg CH4",
)
fig.update_layout(template="plotly_white", yaxis={"categoryorder": "total ascending"})
fig.show()

## 7. Weather, and the two periods that matter

> **Everything above this point runs in seconds. Everything below takes a minute or
> two.** Sections 1-6 are algebra on a one-kilogram basis. From here the notebook
> dispatches a weather record hour by hour, which is the slow part.
>
> **This notebook is deliberately shorter than the real thing:** one training year and
> five evaluation years, against the five and ten every shipped result uses. That is
> purely for speed. The structure below is exactly the shipped procedure; only the
> record lengths are cut. Two consequences worth knowing: a single training year is not
> really a climatology - the "forecast" is just that one year's weather - and a
> single training year under-builds harder than five would, so the sizing gap in
> section 10 is larger here than in the shipped study, not smaller.

The two periods do different jobs, and conflating them is the single easiest way to
flatter the model:

- the **training period** is everything a designer could have known before building. It
  produces the climatology forecast *and* specifies the plant;
- the **evaluation period** is the years the finished plant then has to live through.
  It is the only period that is dispatched.

The forecast is the mean capacity factor for the same calendar day and hour across the
training years, with leap day interpolated. Dispatch sees actual weather one hour at a
time and never receives a future actual value.


In [ ]:
# Shortened for speed: shipped results use 5 training + 10 evaluation years.
TRAINING_YEARS, EVALUATION_YEARS = 1, 5

weather = model.make_synthetic_weather(
    start="2011-01-01", years=TRAINING_YEARS + EVALUATION_YEARS, seed=7)
split = 2011 + TRAINING_YEARS
training = weather.loc[weather.index.year < split]    # what the designer had
actual = weather.loc[weather.index.year >= split]     # what the plant then met
forecast = model.build_climatology_forecast(training, actual)

print(f"training   {training.index[0]:%Y} to {training.index[-1]:%Y}"
      f"  ({len(training):,} h)")
print(f"evaluation {actual.index[0]:%Y} to {actual.index[-1]:%Y}"
      f"  ({len(actual):,} h)")

weather_view = pd.DataFrame({
    "actual CF": actual["capacity_factor"],
    "forecast CF": forecast["capacity_factor"],
}).iloc[:24 * 14]
px.line(weather_view,
        title="Two weeks of actual and climatology-forecast capacity factor").show()


To use cached or live solar-only MERRA-2 data instead, uncomment the next cell. The API token stays in .env; request metadata, never the secret, accompanies the cache.

In [ ]:
# weather_config = model.WeatherConfig(lat=51.5074, lon=-0.1278, latest_year=2025)
# full_profile, source_metadata = model.fetch_solar_profile(weather_config)
# training = full_profile.loc[full_profile.index.year <= 2015]
# actual = full_profile.loc[full_profile.index.year >= 2016]
# forecast = model.build_climatology_forecast(training, actual)

## 8. Solar-to-throughput conversion

For through-night operation, CF_long is the mean of all hours. For limping and hard shutdown, it is the mean over solar hours because methane is scheduled only while the sun is up.

$$\dot m_{nom}=\frac{1000\,P_{farm,MW}\,CF_{long}}
{E_{req}(1+f_{OCP})}.$$

This formula sets nameplate methane throughput. It is not an hourly energy balance; standby, startup, storage losses, and forecast errors are handled in the schedule and dispatch.

In [ ]:
strategy_rows = []
for short_name in ("through_night", "limping", "hard_shutdown"):
    strategy = model.StrategyConfig(short_strategy=short_name, f_ocp=0.20, parallel_reactor_count=4)
    nominal = model.nominal_methane_rate(actual, integrated_energy, plant, strategy)
    daylight = actual["capacity_factor"] > strategy.daylight_cf_cutoff
    cf_long = (
        actual["capacity_factor"].mean()
        if short_name == "through_night"
        else actual.loc[daylight, "capacity_factor"].mean()
    )
    strategy_rows.append({
        "short strategy": short_name,
        "CF_long": cf_long,
        "nominal methane kg/h": nominal,
        "full-load process kW": nominal * e_req,
    })
nominal_table = pd.DataFrame(strategy_rows).set_index("short strategy")
nominal_table

## 9. How strategy changes the hourly load

- **Through night:** constant methane target and continuous hot standby.
- **Limping:** zero methane at night, but both thermal masses remain hot; standby heat continues.
- **Hard shutdown:** zero night production and no standby heat. The plant cools according to
  $T_{t+1}=T_{amb}+(T_t-T_{amb})\exp(-UA\Delta t/C)$, then solar energy pays the reheat load before production restarts.

The next cell exposes each load component rather than only plotting a total.

In [ ]:
week = forecast.iloc[:24 * 7]
schedules = {}
schedule_summary = []
for short_name in ("through_night", "limping", "hard_shutdown"):
    strategy = model.StrategyConfig(short_strategy=short_name, parallel_reactor_count=4)
    nominal = model.nominal_methane_rate(actual, integrated_energy, plant, strategy)
    schedule = model.build_target_schedule(
        week, nominal, integrated_energy, plant, thermal, strategy
    )
    schedules[short_name] = schedule
    schedule_summary.append({
        "strategy": short_name,
        "process energy (kWh)": schedule["process_load_kw"].sum(),
        "standby energy (kWh)": schedule["thermal_load_kw"].sum(),
        "startup energy (kWh)": schedule["startup_load_kwh"].sum(),
        "methane target (kg)": schedule["target_methane_kg_h"].sum(),
    })
pd.DataFrame(schedule_summary).set_index("strategy")

In [ ]:
schedule_plot = pd.concat(
    {
        name: frame[["process_load_kw", "thermal_load_kw", "startup_load_kwh"]]
        for name, frame in schedules.items()
    },
    names=["strategy", "timestamp"],
).reset_index().melt(
    id_vars=["strategy", "timestamp"],
    var_name="load component",
    value_name="kW or kWh in hour",
)
px.line(
    schedule_plot,
    x="timestamp",
    y="kW or kWh in hour",
    color="load component",
    facet_row="strategy",
    title="One-week strategy-dependent process, standby, and startup loads",
    height=750,
).show()

### Thermal inventory implied by tau

Here, tau is the solids cycle time—not a free thermal time constant. Effective hot-solids inventory is stoichiometric mass flow times tau. The inventory heat capacity then controls passive cooling together with UA. The example below uses the hard-shutdown nominal methane rate.

In [ ]:
hard_nominal = nominal_table.loc["hard_shutdown", "nominal methane kg/h"]
cao_inventory, cao_capacity = calc.thermal_inventory_kwh_per_k(
    stoich["CaO_circulating"] * hard_nominal,
    plant.solids_cycle_time_h,
    plant.cao_cp_kwh_per_kg_k,
)
after_one_hour = calc.lumped_cooling_step_k(
    plant.carbonator_temperature_k,
    thermal.ambient_fallback_k,
    thermal.carbonator_ua_kw_per_k,
    cao_capacity,
)
pd.Series({
    "CaO inventory (kg)": cao_inventory,
    "CaO lumped heat capacity (kWh/K)": cao_capacity,
    "initial carbonator temperature (K)": plant.carbonator_temperature_k,
    "temperature after one shutdown hour (K)": after_one_hour,
}, name="value").to_frame()

## 10. Storage sizing, twice, for two different plants

Storage energy comes from the range of a cyclic state-of-charge trajectory: net bus
energy each hour is PV generation minus process, standby and startup load, daily-mean
residuals go to the long store and within-day deviations to the short store. Long-term
installed energy is then scaled by $f_{SOCP}$.

The important structural point is that this happens **twice**, on two different records:

| Case | Sized on | Represents |
| --- | --- | --- |
| Perfect information | the evaluation period | an upper bound nobody can build to |
| Imperfect, faulted, baseline | the **training** record | the plant a designer would actually have specified |

A real designer commits to capacities before the decade they must survive, so the
forecast-driven cases run a plant sized on the earlier period. That plant can turn out
too small, in which case the run reports a cyclic energy deficit rather than failing.

The comparison below sizes on both records so the difference is visible. In the shipped
ten-site study the training-record plant averages about 9% less capital than the
hindsight plant, driven by whether the training window happened to contain a hard
winter.

Expect a **larger** gap here, not a smaller one: a single training year sees a single
winter, so it is an even thinner basis than five years and under-builds harder. That is
the whole mechanism on display - the shorter the design record, the worse the guess.

Power is sized in a separate step: run the dispatch with these energy capacities but
unlimited transfer power, then report the maximum observed charge and discharge.


In [ ]:
sizing_rows = []
for label, record in (("training record", training), ("evaluation period", actual)):
    for short_name in ("through_night", "limping", "hard_shutdown"):
        strategy = model.StrategyConfig(
            short_strategy=short_name, f_ocp=0.20, f_socp_long=1.0,
            parallel_reactor_count=4,
        )
        sizing = model.size_perfect_storage(
            record, integrated_energy, plant=plant, thermal=thermal,
            short_storage=short_storage, long_storage=battery_long_storage,
            strategy=strategy,
        )
        sizing_rows.append({
            "sized on": label,
            "short strategy": short_name,
            "nominal CH4 (kg/h)": sizing.nominal_methane_kg_h,
            "short capacity (kWh)": sizing.short_capacity_kwh,
            "long capacity (kWh)": sizing.long_capacity_kwh,
            "long charge power (kW)": sizing.long_required_charge_power_kw,
            "cyclic feasible": sizing.feasible,
            "balance deficit (MWh/y)": sizing.average_annual_balance_deficit_kwh / 1000,
        })
sizing_table = pd.DataFrame(sizing_rows).set_index(["sized on", "short strategy"])
sizing_table


## 11. Perfect versus imperfect dispatch

`run_case` takes `sizing_profile`: give it the training record and the forecast-driven
cases run the plant that record specified, while perfect information keeps its own.
Omit it and every case is sized on `actual`, which is the older single-plant behaviour
and quietly hands the forecast-driven cases knowledge they could not have had.

Actual installed storage is

$$C_{actual}=C_{perfect}f_{SOCP},$$

with separate short- and long-store factors. Here $f_{SOCP}=1$ installs exactly the required energy capacity, $f_{SOCP}=0.75$ is 25% undersized, and $f_{SOCP}>1$ adds reserve. There is no additional partial-storage multiplier. Installed transfer power is

$$P_{installed}=P_{required}\,f_{POCP},$$

where $P_{required}$ comes from the unlimited-power reference run and $f_{POCP}$ defaults to 1.0. Perfect dispatch uses actual CF as its schedule input. Imperfect dispatch uses the climatology forecast, then responds to actual CF without look-ahead.

Within each hour the priority is essential thermal/startup demand, scheduled production, short storage, long storage, then curtailment. During shortages, planned charging disappears first, then short and long stores discharge, methane production is reduced, and finally the plant shuts down if essential heat cannot be served.

In [ ]:
base_strategy = model.StrategyConfig(
    short_strategy="limping",
    f_ocp=0.20,
    f_socp_long=1.0,
    parallel_reactor_count=4,
)
fault_scenario = model.FaultScenario(seed=0)
case = model.run_case(
    actual, forecast, plant=plant, thermal=thermal, strategy=base_strategy,
    short_storage_template=short_storage,
    long_storage_template=battery_long_storage,
    fault_scenario=fault_scenario,
    sizing_profile=training,   # the forecast-driven cases get the designer's record
    include_baseline=True,     # and the do-nothing reference to measure them against
)

comparison = pd.DataFrame({
    "perfect information": case.perfect.metrics,
    "climatology forecast": case.imperfect.metrics,
    "climatology + seeded faults": case.imperfect_with_faults.metrics,
})
display(comparison.loc[[
    "methane_total_kg",
    "average_annual_methane_shortfall_kg",
    "plant_utilisation",
    "curtailed_energy_kwh",
    "curtailment_fraction",
    "forced_shutdown_hours",
    "max_abs_energy_balance_residual_kwh",
]])
pd.Series(case.imperfect.metadata["storage_power"], name="kW or fraction").to_frame()

In [ ]:
model.build_dispatch_figure(
    case.imperfect_with_faults, "Limping + constant output: seeded faulted dispatch"
).show()

## 12. Strategy comparison

The three current operating strategies use the same weather, plant equations, constant-output long-horizon policy, four parallel Sabatier trains, and independent capacity factors. Each UTC day is planned by strict half-up rounding of continuous train equivalents with no carry-forward. Storage and nominal throughput are re-sized for each case, so the comparison captures the consequences of the thermal operating strategy. Seasonal throttling remains deferred. Forecast error does not mathematically guarantee a higher LCOM; the ratio is reported, not imposed.

In [ ]:
case_rows = []
storage_factor = 1.0
for short_name in ("through_night", "limping", "hard_shutdown"):
    strategy = model.StrategyConfig(
        short_strategy=short_name,
        f_ocp=0.20,
        f_socp_long=storage_factor,
        parallel_reactor_count=4,
    )
    result = model.run_case(
        actual, forecast, plant=plant, thermal=thermal, strategy=strategy,
        short_storage_template=short_storage,
        long_storage_template=battery_long_storage,
    )
    case_rows.append({
        "short strategy": short_name,
        "long policy": strategy.long_strategy,
        "f_SOCP": storage_factor,
        "initial SOC fraction": initial_soc_fraction,
        "methane (kg)": result.imperfect.metrics["methane_total_kg"],
        "utilisation": result.imperfect.metrics["plant_utilisation"],
        "curtailment": result.imperfect.metrics["curtailment_fraction"],
        "shutdown hours": result.imperfect.metrics["forced_shutdown_hours"],
        "required short store (kWh)": result.sizing.short_capacity_kwh,
        "installed short store (kWh)": result.imperfect.metadata["short_storage"]["capacity_kwh"],
        "required long store (kWh)": result.sizing.long_capacity_kwh,
        "installed long store (kWh)": result.imperfect.metadata["long_storage"]["capacity_kwh"],
        "LCOM actual (USD/kg)": result.economics_imperfect["lcom_usd_per_kg_ch4"],
        "LCOM actual/perfect": result.economics_imperfect["relative_prediction_cost_ratio"],
    })
strategy_results = pd.DataFrame(case_rows)
strategy_results

In [ ]:
px.scatter(
    strategy_results,
    x="curtailment",
    y="LCOM actual (USD/kg)",
    size="methane (kg)",
    color="short strategy",
    hover_data=[
        "utilisation", "shutdown hours", "f_SOCP",
        "installed short store (kWh)", "installed long store (kWh)",
    ],
    title="Strategy trade-off: curtailment, methane output, and LCOM",
).show()

## 13. Audit checks and next experiments

Every dispatch records both storage SOCs, actual and forecast CF, production, operating state, temperatures, curtailment, direct hydrogen use, and an hourly energy-balance residual. The residual should remain at floating-point noise.

Useful next edits/tests:

- vary f_ocp and f_socp_long;
- switch a storage template to method="hydrogen" to test direct-H₂-first discharge;
- vary the seeded battery, hydrogen, Sabatier, and DAC fault distributions;
- replace demo weather with a cached 15-year site;
- use save_outputs=True in run_case to write complete case artifacts.



In [ ]:
audit = case.imperfect.hourly[[
    "actual_cf",
    "forecast_cf",
    "methane_kg",
    "short_soc_kwh",
    "long_soc_kwh",
    "curtailed_kwh",
    "carbonator_temperature_k",
    "calciner_temperature_k",
    "energy_balance_residual_kwh",
]]
print(
    "Maximum absolute hourly energy-balance residual:",
    audit["energy_balance_residual_kwh"].abs().max(),
    "kWh",
)
print("Case warnings:", case.imperfect.warnings)
